# NeurIPS 2026 Smart Buildings — Version 8

**Experiment: time-domain event-triggered control with finite-state comfort relays.**

Version 8 deliberately targets a different mechanism from earlier versions: **actuator switching stability**, not thermodynamic mixing (V7), PID (V2–3), future temperature forecasting (V4), comfort-regret prioritization (V5), or learned models with action search (V6).

## Control stages and where to read

| Stage | Method | What it does |
| --- | --- | --- |
| Episode topology | `__init__` | Maps observations, zones, AHUs, and actuators from building metadata |
| Sensor decoding | `_read_zone`, `_schedule_fallback` | Converts normalized zone temperature to Kelvin and validates live comfort bounds |
| Temporal relay | `_run_permission` | Turns equipment on at 0.35 K excursion, off below 0.10 K, with two-step minimum run |
| Zone VAV control | `_control_zone` | Separate cooling dampers and heating/reheat valves from direct comfort errors |
| AHU control | `_control_ahu` | Uses a direct cooling setpoint and peak/mean damper demand for fan pressure |
| Shared hot water | `_control_hot_water` | Runs the water system only when a zone requests reheat |
| Actuator stability | `_ramp` | Caps changes in coil, fan, VAV damper and reheat-valve commands during continuous operation |
| Gateway | `get_action` | Coordinates one inference step; enforces finite float32/int32 actions |
| Episode reset | `reset` | Clears switching latches and previously issued commands |

**Why try it?** Equipment startup and abrupt setpoint changes can waste energy or create control oscillations. V8 tests the hypothesis that modest switching hysteresis plus 5-minute action-rate limits can improve aggregate reward while preserving comfort. It is deliberately **not** a V7 parameter sweep. On an AHU's *first* active step, ramp limits are bypassed for prompt corrective action. Reheat and dampers are always commanded explicitly; normalized zero is not an off state.

**Implementation trade-offs:** The relay is stateful, so `reset()` must run between episodes. There is no attempt to estimate the physical coil/mixing equations or chiller cost. The return-air fraction stays at the maximum command; this intentionally gives up V7's free cooling. The fixed finite state is O(AHUs + action dimensions), and each call is O(AHUs + zones).

## Validation

`python Co_Kaggle/g8/submissions/test_version8.py` executes the actual notebook policy and verifies dynamic layouts, hot/cold/mixed demand, hysteresis, dwell, actuator slew, sensor loss, reset, normalized actions, and inference gateway wiring.

This test establishes **interface compliance and synthetic control behavior only**. No official simulator reward, leaderboard improvement, or 30-scenario evaluation is claimed. Existing best published record is **V7: 0.03947 (lower is better)**. Run a paired full-length evaluation before deciding whether to retain V8.

Submit this notebook directly as a Kaggle code notebook. The last cell invokes the official inference server.


In [ ]:
"""Version 8: event-triggered, slew-limited HVAC control.

A temporal controller: finite-state comfort alarms and bounded changes in
actuator commands reduce chattering. Unlike V7 it does not model mixed air,
choose economizer fractions, or allocate flow using temperature headroom.
Unlike V4 it uses neither trend forecasts nor pre-occupancy predictions.

No optimization, model fitting, or unbounded history. Per-step O(AHUs + zones).
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Thermostatic dispatch with dwell protection and actuator ramp limits.

    Stateless comfort readings become short-lived AHU on/off latches. The
    action interface is continuous float32 [-1, 1], discrete int32 {0, 1}.
    """

    # Kelvin comfort thresholds; start requires stronger evidence than stop.
    START_ERROR_K = 0.35
    STOP_ERROR_K = 0.10
    MIN_RUN_STEPS = 2  # Five-minute samples, including the starting sample.
    ZONE_DEMAND_K = 0.10

    # Action changes per five minutes, in normalized action units.
    MAX_AIR_TEMP_STEP = 0.35  # 3.5 K in the 285..305 K command range.
    MAX_PRESSURE_STEP = 0.35  # 350 Pa in the 0..2000 Pa command range.
    MAX_DAMPER_STEP = 0.50  # 25 percentage points.
    MAX_VALVE_STEP = 0.60  # 30 percentage points.

    MIN_VAV_FRACTION = 0.10

    def __init__(self, building_meta_data, action_spec):
        """Index observations and actions once, never during each time step."""
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(s): i for i, s in enumerate(meta["observation_field_names"])}
        self.supply_indices = list(map(int, meta["temp_indices"]))
        self.pressure_indices = list(map(int, meta["pressure_indices"]))
        self.recirc_indices = list(map(int, meta["recirculation_indices"]))
        self.water_indices = list(map(int, meta["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_supply = self.water_indices[0] if self.water_indices else None
        self.water_pressure = (self.pressure_indices[self.ahu_count]
                               if len(self.pressure_indices) > self.ahu_count else None)
        self.water_run = self.n_discrete - 1

        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        self.morning = float(meta["morning_start_hour"])
        self.evening = float(meta["evening_start_hour"])
        normalizer = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalizer.get("sample_mean", 0.0))
        self.zone_std = float(normalizer.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        def unit_order(key):
            match = re.search(r"\d+", str(key))
            return (int(match.group()) if match else -1, str(key))

        self.ahus = []
        assignments = meta["ahu_zone_assignments"]
        indices = meta["zone_action_indices"]
        for i, identifier in enumerate(sorted(assignments, key=unit_order)[:self.ahu_count]):
            zones = []
            for room in assignments[identifier]:
                vav = str(room).replace("room_", "VAV_")
                temp = self.fields.get(vav + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                actuators = indices.get(vav, {})
                zones.append({
                    "temperature": temp,
                    "low": self.fields.get(vav + "_zone_air_heating_temperature_setpoint_sensor"),
                    "high": self.fields.get(vav + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": actuators.get("supply_air_damper_percentage_command"),
                    "valve": actuators.get("reheat_valve_percentage_command"),
                })
            self.ahus.append({
                "zones": zones,
                "run": i,
                "supply": self.supply_indices[i],
                "pressure": (self.pressure_indices[i]
                             if i < len(self.pressure_indices) else None),
                "recirc": (self.recirc_indices[i]
                           if i < len(self.recirc_indices) else None),
            })
        self.reset()

    def reset(self):
        """Clear all temporal state at an episode boundary."""
        self.running = [False] * len(self.ahus)
        self.hold = [0] * len(self.ahus)
        self.last_continuous = None

    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, fraction):
        """Convert physical 0..1 actuator opening to normalized action."""
        return 2.0 * cls._clamp(fraction, 0.0, 1.0) - 1.0

    @staticmethod
    def _temperature_action(kelvin):
        """Supply air 285..305 K maps to -1..1."""
        return float(np.clip((kelvin - 295.0) / 10.0, -1.0, 1.0))

    def _value(self, obs, index):
        """Read an optional sensor without indexing outside the observation."""
        if index is None or not 0 <= index < len(obs):
            return float("nan")
        return float(obs[index])

    def _schedule_fallback(self, obs):
        """Use metadata comfort/eco windows only when live setpoints are invalid."""
        def clock(sin_name, cos_name, period):
            sine = self._value(obs, self.fields.get(sin_name))
            cosine = self._value(obs, self.fields.get(cos_name))
            if not (np.isfinite(sine) and np.isfinite(cosine)):
                return None
            return float(np.arctan2(sine, cosine) % (2 * np.pi)) * period / (2 * np.pi)

        hour = clock("hod_sin_000", "hod_cos_000", 24.0)
        day = clock("dow_sin_000", "dow_cos_000", 7.0)
        weekday = day is not None and int(day) < 5
        occupied = (weekday and hour is not None
                    and self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zone(self, obs, zone, fallback):
        """Decode one zone (Kelvin); return None for an invalid temperature.

        The temperature observation is standardized. Setpoint observations
        are already in Kelvin and take precedence over schedule fallback.
        """
        raw = self._value(obs, zone["temperature"])
        temperature = raw * self.zone_std + self.zone_mean
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._value(obs, zone["low"])
        high = self._value(obs, zone["high"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {
            "actions": zone,
            "temp": temperature,
            "hot": max(0.0, temperature - high),
            "cold": max(0.0, low - temperature),
        }

    def _make_idle_actions(self):
        """A zero action is half-open, so explicitly set idle commands."""
        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)
        discrete = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply_indices:
            continuous[index] = 1.0  # Warmest: cooling coil effectively idle.
        for index in self.recirc_indices:
            continuous[index] = 1.0  # Max return air, physically capped at 70%.
        return continuous, discrete

    def _run_permission(self, ahu_index, statuses):
        """Two-threshold finite-state relay with a short minimum run time.

        A sensor outage turns off the AHU rather than retaining stale heat or
        cooling requests. No error integral or long-lived inferred model.
        """
        if not statuses:
            self.running[ahu_index] = False
            self.hold[ahu_index] = 0
            return False

        worst = max(max(z["hot"], z["cold"]) for z in statuses)
        if not self.running[ahu_index]:
            if worst < self.START_ERROR_K:
                return False
            self.running[ahu_index] = True
            self.hold[ahu_index] = self.MIN_RUN_STEPS - 1
            return True

        if self.hold[ahu_index] > 0:
            self.hold[ahu_index] -= 1
            return True
        if worst >= self.STOP_ERROR_K:
            return True

        self.running[ahu_index] = False
        return False

    def _ramp(self, command, index, maximum_change):
        """Limit normalized command jumps while the same AHU keeps running."""
        if index is None or not 0 <= index < self.n_continuous:
            return
        if self.last_continuous is None:
            return
        previous = float(self.last_continuous[index])
        command[index] = self._clamp(
            float(command[index]), previous - maximum_change,
            previous + maximum_change)

    def _control_zone(self, status, command, was_running, heating_load):
        """Map comfort excursions to independently modulated VAV actuators.

        Cooling uses a damper; heating uses both VAV airflow and a reheat
        valve. This deliberately avoids V7's thermodynamic headroom formula.
        Returns the requested damper fraction for fan coordination.
        """
        indices = status["actions"]
        hot, cold = status["hot"], status["cold"]
        damper, valve = self.MIN_VAV_FRACTION, 0.0

        if hot >= self.ZONE_DEMAND_K:
            damper = self._clamp(0.16 + 0.18 * hot, 0.16, 0.98)
        elif cold >= self.ZONE_DEMAND_K:
            damper = self._clamp(0.24 + 0.10 * cold, 0.24, 0.85)
            valve = self._clamp(0.15 + 0.24 * cold, 0.0, 1.0)
            if indices["valve"] is not None:
                heating_load.append((cold, valve))

        damper_index, valve_index = indices["damper"], indices["valve"]
        if damper_index is not None and 0 <= damper_index < self.n_continuous:
            command[damper_index] = self._fraction_action(damper)
            if was_running:
                self._ramp(command, damper_index, self.MAX_DAMPER_STEP)
        if valve_index is not None and 0 <= valve_index < self.n_continuous:
            command[valve_index] = self._fraction_action(valve)
            if was_running:
                self._ramp(command, valve_index, self.MAX_VALVE_STEP)
        return damper

    def _control_ahu(self, ahu_index, ahu, statuses, command, discrete, heating_load):
        """Apply the mode relay, zone demand commands, and shared fan reset."""
        was_running = self.running[ahu_index]
        if not self._run_permission(ahu_index, statuses):
            return

        discrete[ahu["run"]] = 1
        maximum_hot = max(z["hot"] for z in statuses)
        # The AHU is cooling-only: warmest supply for heat-only demand.
        coil_k = (self._clamp(297.0 - 1.30 * maximum_hot, 285.0, 305.0)
                  if maximum_hot >= self.ZONE_DEMAND_K else 305.0)
        command[ahu["supply"]] = self._temperature_action(coil_k)
        if was_running:
            self._ramp(command, ahu["supply"], self.MAX_AIR_TEMP_STEP)

        # Keep fixed maximum return air; this experiment does not optimize
        # the economizer or infer mixed-air temperature.
        if ahu["recirc"] is not None:
            command[ahu["recirc"]] = 1.0

        dampers = [
            self._control_zone(status, command, was_running, heating_load)
            for status in statuses
        ]
        if ahu["pressure"] is not None:
            # Use max demand to protect one extreme zone among neutral zones;
            # the mean adds moderate pressure when many rooms need airflow.
            mean = sum(dampers) / len(dampers)
            peak = max(dampers)
            pressure_pa = self._clamp(
                220.0 + 1020.0 * max(mean, 0.70 * peak), 200.0, 1700.0)
            command[ahu["pressure"]] = self._clamp(
                pressure_pa / 1000.0 - 1.0, -1.0, 1.0)
            if was_running:
                self._ramp(command, ahu["pressure"], self.MAX_PRESSURE_STEP)

    def _control_hot_water(self, heating_load, command, discrete):
        """Coordinate the shared water plant only for actual reheat demand."""
        if not heating_load or not 0 <= self.water_run < self.n_discrete:
            return

        discrete[self.water_run] = 1
        strongest = max(cold for cold, _ in heating_load)
        mean_valve = sum(valve for _, valve in heating_load) / len(heating_load)
        # Direct demand reset; no V7 outdoor-air temperature compensation.
        water_k = self._clamp(321.0 + 4.0 * strongest, 320.0, 348.0)
        if self.water_supply is not None:
            command[self.water_supply] = self._fraction_action(
                (water_k - 310.0) / 40.0)
        if self.water_pressure is not None:
            command[self.water_pressure] = self._fraction_action(
                self._clamp(0.35 + 0.55 * mean_valve, 0.35, 0.95))

    def get_action(self, obs):
        """One five-minute decision; finite temporal state, linear work.

        Stages: decode sensors, decide AHU run state, set zone actuators,
        coordinate water plant, sanitize and save prior normalized commands.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._make_idle_actions()
        fallback = self._schedule_fallback(obs)
        heating_load = []

        for ahu_index, ahu in enumerate(self.ahus):
            statuses = [self._read_zone(obs, zone, fallback)
                        for zone in ahu["zones"]]
            statuses = [status for status in statuses if status is not None]
            self._control_ahu(
                ahu_index, ahu, statuses, continuous, discrete, heating_load)

        self._control_hot_water(heating_load, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        self.last_continuous = continuous.copy()
        return {"continuous_action": continuous, "discrete_action": discrete}


In [ ]:
# Official Kaggle evaluation gateway. Do not replace with a local CSV writer.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
